# RanPAC Critique B · Stress-test *Class Imbalance* của Hồi quy Ridge ở Phase 2

Notebook này thuộc phần **Critical Analysis & Empirical Demonstration** trên **Split CIFAR-100** (`seed = 1993`), tận dụng trực tiếp **FeatureCache ViT-B/16 đã có sẵn** (0 giây trích đặc trưng GPU) để chứng minh điểm yếu toán học của hàm mục tiêu Phase 2 trong RanPAC:

### Cơ chế gây lỗi dưới dữ liệu mất cân bằng lớp
RanPAC tích lũy ma trận Gram và tương quan chéo không trọng số:
$$G = \sum_{i=1}^{N} h_i h_i^\top, \qquad C_{:, c} = \sum_{y_i = c} h_i, \qquad W_{\text{out}} = (G + \lambda I)^{-1} C$$
Khi số mẫu $n_c$ chênh lệch mạnh giữa lớp đa số (*Head*, $n_{\text{head}} = 500$) và lớp thiểu số (*Tail*, $n_{\text{tail}} = 500 / \rho$):
1. Cột $C_{:, c}$ tỷ lệ tuyến tính với $n_c$, khiến chuẩn cột trọng số $\|W_{\text{out}}[:, c]\|_2$ của lớp ít mẫu bị co nhỏ nghiêm trọng.
2. Phổ của $G + \lambda I$ bị chi phối hoàn toàn bởi phương sai của các lớp đa số, đẩy biên quyết định đè lên vùng không gian của lớp thiểu số.

### Hai kịch bản kiểm chứng & Bốn bộ phân loại đối chiếu
1. **Kịch bản 1 — Inter-task Step Imbalance ($\rho \in \{1, 5, 10, 20, 50\}$):** Tasks 1–5 dồi dào dữ liệu (`500` mẫu/lớp), Tasks 6–10 khan hiếm dữ liệu (`500/ρ` mẫu/lớp).
2. **Kịch bản 2 — Intra-task Long-Tail ($\rho \in \{10, 50\}$):** Phân phối đuôi dài hàm mũ ngay bên trong từng task, kèm kiểm chứng cả `Task-CV λ` để chứng minh việc chọn $\lambda$ theo task **không cứu được** các lớp thiểu số.

| Bộ phân loại | Đặc tính dưới Class Imbalance |
| --- | --- |
| `ncm` | **NCM cosine:** chuẩn hóa trung bình theo từng lớp ($\mu_c = s_c / n_c$) $\Rightarrow$ gần như bất biến với tỷ lệ mất cân bằng $\rho$. |
| `ridge` | **Ridge 768-d (không RP):** giải MSE không trọng số trực tiếp trên đặc trưng gốc. |
| `ridge-rp` | **RanPAC Phase 2 (`M = 10000`, ReLU):** cấu hình chuẩn của bài báo — sụt giảm mạnh trên lớp thiểu số khi $\rho$ tăng. |
| `cb-ridge-rp` | **Class-Balanced Ridge + RP (Đối chứng khắc phục):** dùng cùng ma trận $W$, cân bằng lại mỗi mẫu theo $w_c = N_{\max} / n_c$, khôi phục độ chính xác lớp thiểu số. |


## 1. Cài đặt thư viện và kết nối Google Drive

Đặt `ranpac_core.py`, `ranpac_data.py`, `ranpac_colab.py`, `ranpac_full.py`, `ranpac_critique.py` và `test_ranpac_core.py` trong `MyDrive/RanPAC_implement`. Nếu trên Drive đã có sẵn cache embedding của các notebook `01`–`05`, notebook này có thể chạy ngay cả trên **CPU runtime** hoặc **GPU runtime** trong ~1–2 phút.


In [ ]:
%pip install -q timm==1.0.22 wandb==0.31.0


In [ ]:
import sys
import unittest
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from google.colab import drive

drive.mount('/content/drive')
SOURCE_DIR = Path('/content/drive/MyDrive/RanPAC_implement')
WORK_DIR = Path('/content/drive/MyDrive/ranpac-runs')
DATA_ROOT = '/content/ranpac-data'

required = (
    'ranpac_core.py', 'ranpac_data.py', 'ranpac_colab.py',
    'ranpac_full.py', 'ranpac_critique.py', 'test_ranpac_core.py',
)
CODE_DIR = SOURCE_DIR / 'src' if (SOURCE_DIR / 'src' / 'ranpac_core.py').is_file() else SOURCE_DIR
missing = [name for name in required if not (CODE_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f'Thiếu {", ".join(missing)} trong {CODE_DIR}')
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from ranpac_core import NUM_TASKS, SEED, split_class_order
from ranpac_critique import run_imbalance_experiment
from ranpac_data import FeatureCache, atomic_output, feature_config


In [ ]:
suite = unittest.defaultTestLoader.loadTestsFromName('test_ranpac_core')
test_result = unittest.TextTestRunner(verbosity=2).run(suite)
if not test_result.wasSuccessful():
    raise AssertionError('Kiểm tra unit test trên CPU chưa đạt.')


## 2. Cấu hình lưới mất cân bằng và nạp Feature Cache dùng chung

Lưu ý kỹ thuật: khóa `FeatureCache` chuẩn của dự án dùng `feature_device_type = 'cuda'`, nên cell dưới khởi tạo khóa với `torch.device('cuda')` để khớp trực tiếp thư mục cache đã có trên Drive (nếu chưa có cache mới cần GPU runtime để trích một lần).


In [ ]:
WANDB_ENTITY = 'pnp123'
WANDB_PROJECT = 'RanPAC-reproduction'
WANDB_MODE = 'online'          # 'online', 'offline', hoặc 'disabled'
RUN_LABEL = 'imb-01'

M = 10_000
FIXED_LAMBDA = 100.0
BATCH_SIZE = 48
NUM_WORKERS = 2
STATS_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

METHODS = ('ncm', 'ridge', 'ridge-rp', 'cb-ridge-rp')
STEP_RATIOS = (1, 5, 10, 20, 50)
LONG_TAIL_RATIOS = (10, 50)
RUN_TASK_CV_CHECK = True       # Kiểm tra thêm Task-CV lambda ở rho = 10, 50

extract_dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cache = FeatureCache(
    WORK_DIR / 'embeddings', DATA_ROOT,
    feature_config(split_class_order(SEED), torch.device('cuda')),
    extract_dev, BATCH_SIZE, NUM_WORKERS,
)
missing_shards = [
    (sp, t) for t in range(1, NUM_TASKS + 1) for sp in ('train', 'test')
    if not (cache.root / f'{sp}-task-{t:02}.pt').is_file()
]
if missing_shards and not torch.cuda.is_available():
    raise RuntimeError('Chưa có cache trên Drive; hãy chọn GPU runtime để trích lần đầu.')

train_shards_full = [cache.load('train', t)[:2] for t in range(1, NUM_TASKS + 1)]
test_shards = [cache.load('test', t)[:2] for t in range(1, NUM_TASKS + 1)]
print(f'Đã nạp đủ 10 train shards + 10 test shards từ {cache.root} (stats_device={STATS_DEVICE}).')


## 3. Chạy Sweep Class Imbalance (Step Imbalance & Long-Tail)


In [ ]:
all_summaries = []

print('=== Phần 1: Inter-task Step Imbalance (Tasks 1..5 = 500/lớp, Tasks 6..10 = 500/ρ) ===')
for ratio in STEP_RATIOS:
    for method in METHODS:
        out = run_imbalance_experiment(
            method, 'step', ratio, train_shards_full, test_shards, WORK_DIR,
            projection_dim=M, fixed_lambda=FIXED_LAMBDA, stats_device=STATS_DEVICE,
            seed=SEED, run_label=RUN_LABEL, entity=WANDB_ENTITY,
            project=WANDB_PROJECT, wandb_mode=WANDB_MODE,
        )
        all_summaries.append(out['summary'])

print('\n=== Phần 2: Intra-task Exponential Long-Tail ===')
for ratio in LONG_TAIL_RATIOS:
    for method in METHODS:
        out = run_imbalance_experiment(
            method, 'long_tail', ratio, train_shards_full, test_shards, WORK_DIR,
            projection_dim=M, fixed_lambda=FIXED_LAMBDA, stats_device=STATS_DEVICE,
            seed=SEED, run_label=RUN_LABEL, entity=WANDB_ENTITY,
            project=WANDB_PROJECT, wandb_mode=WANDB_MODE,
        )
        all_summaries.append(out['summary'])

if RUN_TASK_CV_CHECK:
    print('\n=== Phần 3: Kiểm chứng RanPAC chuẩn với Task-CV λ dưới Step Imbalance ===')
    for ratio in (10, 50):
        out_cv = run_imbalance_experiment(
            'ridge-rp', 'step', ratio, train_shards_full, test_shards, WORK_DIR,
            projection_dim=M, fixed_lambda=None, stats_device=STATS_DEVICE,
            seed=SEED, run_label=RUN_LABEL, entity=WANDB_ENTITY,
            project=WANDB_PROJECT, wandb_mode=WANDB_MODE,
        )
        all_summaries.append(out_cv['summary'])


## 4. Phân tích định lượng và Trực quan hóa sự sụp đổ trên lớp thiểu số

- **Biểu đồ trái:** Độ chính xác trên nhóm lớp thiểu số (`Tail Accuracy`, Tasks 6–10) theo tỷ lệ mất cân bằng $\rho$. Khi $\rho$ tăng lên `20–50`, `RanPAC (ridge-rp)` rơi mạnh xuống dưới cả `NCM cosine` vốn rất đơn giản, trong khi `CB-Ridge + RP` khôi phục lại hiệu năng.
- **Biểu đồ giữa:** Tỷ số chuẩn cột trọng số $\|W_{\text{out}}[:, \text{head}]\|_2 / \|W_{\text{out}}[:, \text{tail}]\|_2$ giải thích trực tiếp nguyên nhân cơ học.
- **Biểu đồ phải:** So sánh `Head` vs `Tail` ở kịch bản `Long-Tail (ρ = 50)`.


In [ ]:
df_imb = pd.DataFrame(all_summaries)
summary_dir = WORK_DIR / 'sweeps' / f'class-imbalance-s{SEED}-{RUN_LABEL}'
with atomic_output(summary_dir / 'class_imbalance_summary.csv') as tmp:
    df_imb.to_csv(tmp, index=False)

cols_show = [
    'imbalance_mode', 'imbalance_ratio', 'method', 'lambda_mode',
    'a10_all_pct', 'head_acc_pct', 'tail_acc_pct', 'head_tail_gap_pp',
    'tail_pred_share_pct', 'weight_norm_ratio_head_to_tail',
]
display(df_imb[cols_show].round(2))

df_step = df_imb[(df_imb['imbalance_mode'] == 'step') & (df_imb['lambda_mode'] != 'task_cv')]
palette = {
    'ncm': ('#64748b', 'o--', 'NCM cosine (count-invariant)'),
    'ridge': ('#f59e0b', 's-.', 'Ridge 768-d (unweighted)'),
    'ridge-rp': ('#dc2626', 'D-', 'RanPAC RP M=10000 (unweighted)'),
    'cb-ridge-rp': ('#16a34a', '^-', 'CB-Ridge + RP M=10000 (balanced)'),
}

fig, axes = plt.subplots(1, 3, figsize=(16, 4.6), layout='constrained')

# Panel 1: Tail Accuracy vs Imbalance Ratio (Step mode)
ax1 = axes[0]
for m in METHODS:
    sub = df_step[df_step['method'] == m].sort_values('imbalance_ratio')
    color, style, label = palette[m]
    ax1.plot(sub['imbalance_ratio'], sub['tail_acc_pct'], style, color=color, linewidth=2.0, label=label)
ax1.set_xscale('log')
ax1.set_xticks(STEP_RATIOS)
ax1.set_xticklabels([f'{r}:1' for r in STEP_RATIOS])
ax1.set_xlabel('Tỷ lệ mất cân bằng ρ (Head : Tail)')
ax1.set_ylabel('Tail Classes Accuracy (%)')
ax1.set_title('Độ chính xác nhóm lớp ít mẫu (Tasks 6–10)')
ax1.grid(True, alpha=0.3)
ax1.legend(fontsize=8.5)

# Panel 2: Head-to-Tail Classifier Weight Norm Ratio
ax2 = axes[1]
for m in ('ncm', 'ridge', 'ridge-rp', 'cb-ridge-rp'):
    sub = df_step[df_step['method'] == m].sort_values('imbalance_ratio')
    color, style, label = palette[m]
    ax2.plot(sub['imbalance_ratio'], sub['weight_norm_ratio_head_to_tail'], style,
             color=color, linewidth=2.0, label=m)
ax2.set_xscale('log')
ax2.set_xticks(STEP_RATIOS)
ax2.set_xticklabels([f'{r}:1' for r in STEP_RATIOS])
ax2.set_xlabel('Tỷ lệ mất cân bằng ρ')
ax2.set_ylabel('||W_head||₂ / ||W_tail||₂')
ax2.set_title('Độ lệch chuẩn cột trọng số phân loại')
ax2.grid(True, alpha=0.3)
ax2.legend(fontsize=8.5)

# Panel 3: Long-Tail (rho = max(LONG_TAIL_RATIOS)) Head vs Tail comparison
ax3 = axes[2]
target_lt = max(LONG_TAIL_RATIOS)
df_lt = df_imb[(df_imb['imbalance_mode'] == 'long_tail') & (df_imb['imbalance_ratio'] == target_lt)]
df_lt = df_lt.set_index('method').loc[list(METHODS)].reset_index()
x = np.arange(len(df_lt))
w = 0.27
ax3.bar(x - w, df_lt['head_acc_pct'], w, label='Head classes', color='#3b82f6')
ax3.bar(x, df_lt['a10_all_pct'], w, label='Overall A[10]', color='#64748b')
ax3.bar(x + w, df_lt['tail_acc_pct'], w, label='Tail classes', color='#ef4444')
ax3.set_xticks(x)
ax3.set_xticklabels(df_lt['method'], rotation=12)
ax3.set_ylabel('Accuracy (%)')
ax3.set_title(f'Intra-task Long-Tail (ρ = {target_lt}:1)')
ax3.grid(axis='y', alpha=0.3)
ax3.legend(fontsize=8.5)

fig_path = summary_dir / 'class_imbalance_analysis.png'
with atomic_output(fig_path) as tmp:
    fig.savefig(tmp, dpi=180, format='png')
plt.show()
print(f'Đã lưu bảng và biểu đồ tại: {summary_dir}')
